<p style="padding: 10px; border: 1px solid black;">
<img src="../common/images/mlu-logo.png" alt="drawing" width="400"/> <br/>
<div style="background-image: linear-gradient(145deg, rgba(35, 47, 62, 1) 0%, rgba(51, 0, 102, 1) 40%, rgba(223, 42, 93, 1) 60%, rgba(124, 90, 237, 1) 85%, rgba(124, 232, 244, 1) 100%); padding: 20px; border-radius: 10px; text-align: center; margin-bottom: 30px;">
    <h1 style="color: white; margin: 0;">MLU: Application of Deep Learning to Text and Image Data</h1>
    <h2 style="color: white; margin-top: 15px;">Fine-Tuning ConvNeXt</h2>
</div>

<!-- Compact Lab Introduction with Activity/Challenge Explanation -->
<div style="background-color: #F8F9F9; padding: 15px; border-radius: 5px; margin: 20px 0;">
    <h4 style="color: #2E4053; margin-top: 0;">About This Lab</h4>
    <p>Throughout this lab, you will encounter two types of interactive elements:</p>
    <table style="width: 100%; border-collapse: collapse; margin: 15px 0;">
        <tr>
            <td style="text-align: center; padding: 10px; width: 50%;">
                <img src="../common/images/mlu-activity.png" alt="Activity" width="125"/>
            </td>
            <td style="text-align: center; padding: 10px; width: 50%;">
                <img src="../common/images/mlu-challenge.png" alt="Challenge" width="125"/>
            </td>
        </tr>
        <tr>
            <td style="text-align: center; padding: 10px; background-color: #EBF5FB;">
                <p>No coding is needed for an activity. You try to understand a concept, <br/>answer questions, or run a code cell.</p>
            </td>
            <td style="text-align: center; padding: 10px; background-color: #FEF9E7;">
                <p>Challenges are where you test your understanding by implementing something new or taking a short quiz.</p>
            </td>
        </tr>
    </table>
    <p>Please work through this notebook from top to bottom to avoid errors due to missing code or context.</p>
</div>

<!-- Table of Contents with All Section Levels -->
<div style="background-color: #f2f0fc; padding: 15px; border-radius: 5px; margin-bottom: 30px;">
    <h2 style="color: #2f1381; border-bottom: 1px solid #2f1381; padding-bottom: 5px;">Table of Contents</h2>
    <p><a href="#section1" style="color: #2f1381; font-weight: bold; text-decoration: none;">1. Loading and transforming the dataset</a></p>
    <p><a href="#section2" style="color: #2f1381; font-weight: bold; text-decoration: none;">2. Fine-tuning the model</a></p>
    <p><a href="#section3" style="color: #2f1381; font-weight: bold; text-decoration: none;">3. Testing and visualizations</a></p>
</div>

<!-- Section Header -->
<div id="section1" style="border-left: 5px solid #2f1381; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2f1381;">1. Loading and transforming the dataset</h2>
</div>

You will be fine-tuning a ConvNeXt model to work with the Materials in Context Database (MINC). To start, use the same steps as the previous lab to use transform functions and create data loaders to load the data.

Reference: Sean Bell, Paul Upchurch, Noah Snavely, and Kavita Bala. "Material Recognition in the Wild with the Materials in Context Database." *Computer Vision and Pattern Recognition (CVPR)*, April 2015. https://arxiv.org/abs/1412.0623.

In [ ]:
# Remove conflicting packages that are not used by this notebook
!pip uninstall -y -q fastai autogluon-multimodal autogluon-timeseries torchtext timm 2>/dev/null || true
# Install libraries
!pip install -U -q -r requirements.txt

Import the PyTorch packages and modules. 

In [ ]:
import os
import matplotlib.pyplot as plt
import torch, torchvision
from PIL import Image
from torch import nn
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torch.optim import SGD
from torchvision.models import convnext_base, convnext_tiny

In [ ]:
# Transform the images
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0,0,0), std=(1,1,1))
])

# Create the data loaders
batch_size = 16
path = '../data/minc-2500'
train_path = os.path.join(path, 'train')
val_path = os.path.join(path, 'val')
test_path = os.path.join(path, 'test')

train_loader = DataLoader(
    ImageFolder(train_path, transform=transform),
    batch_size=batch_size, shuffle=True)

validation_loader = DataLoader(
    ImageFolder(val_path, transform=transform),
    batch_size=batch_size, shuffle=False)

test_loader = DataLoader(
    ImageFolder(test_path, transform=transform),
    batch_size=batch_size, shuffle=False)

<!-- Section Header -->
<div id="section2" style="border-left: 5px solid #2f1381; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2f1381;">2. Fine-tuning the model</h2>
</div>

You can start with a base model and fine-tune it to improve its performance with the dataset that you are using. This helps you to create a better model using fewer resources.

You will take the following steps:
1. Start with a ConvNeXt model that was trained with the ImageNet dataset as the base model.
1. Import the `convnext_tiny` model with the `IMAGENET1K_V1` weights.
1. The original ConvNeXt model was trained for 1,000 categories. Update the last layer with a dense layer that has the same number of classes for the problem that you want to solve (6, in this case).

In [ ]:
classes = 6

# Use GPU resource if available; otherwise use CPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

net = convnext_tiny(weights="IMAGENET1K_V1").to(device)

# Print the network to find out how to access the last layer
print(net)

<!-- Tip Box -->
<div style="background-color: #E8F8F5; border-left: 5px solid #1ABC9C; padding: 15px; border-radius: 5px; margin: 20px 0;">
    <p style="color: #16A085; margin: 0;"><strong>Tip:</strong> Read through the output from the last command. The last layer (a linear layer) is under the __classifier__ part of the network. You can access this layer by using `net.classifier[2]`.</p>
</div>

Now that you have a classifier layer to work from, you need to create a new linear layer that has the same number of inputs and has a preselected number of outputs (6 here).

In [ ]:
num_ftrs = net.classifier[2].in_features

# Update the classifier so that it uses the classes that are defined for your problem
net.classifier[2] = nn.Linear(num_ftrs, classes)

<!-- Activity Box -->
<div style="background-color: #EBF5FB; border-left: 5px solid #3498DB; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../common/images/mlu-activity.png" alt="Activity" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #2874A6; margin-top: 0;">Activity: Train the model</h4>
        <p>Now, run the cell below to train the model. This will take a few minutes to complete.</p>
    </div>
</div>

In [ ]:
%%time
epochs = 5
learning_rate = 0.001
criterion = nn.CrossEntropyLoss()

optimizer = SGD(net.parameters(), lr=learning_rate)

def calculate_accuracy(output, label):
    """Calculate the accuracy of the trained network. 
    output: (batch_size, num_output) float32 tensor
    label: (batch_size, ) int32 tensor """

    return (output.argmax(axis=1) == label.float()).float().mean()

for epoch in range(epochs):
    net = net.to(device) # may be redundant

    train_loss, val_loss, train_acc, valid_acc = 0., 0., 0., 0.

    # Training loop
    # This loop trains the neural network (weights are updated)
    net.train() # Activate training mode
    for data, label in train_loader:
        # Zero the parameter gradients
        optimizer.zero_grad()
        # Put data and label to the correct device
        data = data.to(device)
        label = label.to(device)
        # Make forward pass
        output = net(data)
        # Calculate loss
        loss = criterion(output, label)
        # Make backward pass (calculate gradients)
        loss.backward()
        # Accumulate training accuracy and loss
        train_acc += calculate_accuracy(output, label).item()
        train_loss += loss.item()
        # Update weights
        optimizer.step()

    # Validation loop
    # This loop tests the trained network on the validation dataset
    # No weight updates here
    # torch.no_grad() reduces memory usage when not training the network
    net.eval() # Activate evaluation mode
    with torch.no_grad():
        for data, label in validation_loader:
            data = data.to(device)
            label = label.to(device)
            # Make forward pass with the trained model so far
            output = net(data)
            # Accumulate validation accuracy and loss
            valid_acc += calculate_accuracy(output, label).item()
            val_loss += criterion(output, label).item()x

    # Take averages
    train_loss /= len(train_loader)
    train_acc /= len(train_loader)
    val_loss /= len(validation_loader)
    valid_acc /= len(validation_loader)

    print("Epoch %d: train loss %.3f, train acc %.3f, val loss %.3f, val acc %.3f" % (
        epoch+1, train_loss, train_acc, val_loss, valid_acc))

<!-- Section Header -->
<div id="section3" style="border-left: 5px solid #2f1381; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2f1381;">3. Testing and visualizations</h2>
</div>

Now that you have created a model, you need to validate the model's predictions.

The following code cell defines the `show_images` function. It will show the sample images and the model prediction together.

In [ ]:
def show_images(imgs, num_rows, num_cols, titles=None, scale=1.5):
    """Plot a list of images."""
    figsize = (num_cols * scale, num_rows * scale)
    _, axes = plt.subplots(num_rows, num_cols, figsize=figsize)
    axes = axes.flatten()
    for i, (ax, img) in enumerate(zip(axes, imgs)):
        ax.imshow(img.permute(1,2,0).numpy())
        ax.axes.get_xaxis().set_visible(False)
        ax.axes.get_yaxis().set_visible(False)
        if titles:
            ax.set_title(titles[i])
    return axes

<!-- Activity Box -->
<div style="background-color: #EBF5FB; border-left: 5px solid #3498DB; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../common/images/mlu-activity.png" alt="Activity" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #2874A6; margin-top: 0;">Activity: Test the model</h4>
        <p>Now that you have trained a model and created a function to test predictions, you are ready to load the test dataset, make predictions, and evaluate the predictions.</p>
        <p>Run the cell below to see the model's predictions on test images.</p>
    </div>
</div>

In [ ]:
random_test_dataset = ImageFolder(test_path,
                                  transform=transform)
random_test_sample = DataLoader(random_test_dataset,
                                batch_size=2*8, shuffle=False)

net.eval() # Activate eval mode (don't use dropouts etc.)
for data, label in random_test_sample:
    show_images(data, 2, 8);
    data = data.to(device)
    pred = net(data)
    print(pred.argmax(axis=1))
    break

<!-- Important Note -->
<div style="background-color: #FDEDEC; border-left: 5px solid #E74C3C; padding: 15px; border-radius: 5px; margin: 20px 0;">
    <p style="color: #C0392B; margin: 0;"><strong>Important:</strong> It shows 16 sample images in the test set above. The 16 values in the tensor are the prediction results for these 16 images. The labels are brick(0), carpet(1), food(2), mirror(3), sky(4), and water(5).</p>
</div>

<!-- Challenge Box -->
<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../common/images/mlu-challenge.png" alt="Challenge" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #B7950B; margin-top: 0;">Challenge: Try different ConvNeXt models</h4>
        <p>ConvNeXt comes in different sizes, including <code>convnext_small</code>, <code>convnext_base</code>, and <code>convnext_large</code>.</p>
        <p><strong>Your task:</strong> Pick one of these models, and use the following code cell to retrain your model. You only need to import the new model and replace <code>convnext_tiny</code> with <code>convnext_small</code>, <code>convnext_base</code>, or <code>convnext_large</code>, and retrain the model.</p>
    </div>
</div>

In [ ]:
############### CODE HERE ###############

from torchvision.models import convnext_small
from torchvision.models import convnext_base
from torchvision.models import convnext_large

net = convnext_small(weights="IMAGENET1K_V1").to(device)
net = convnext_base(weights="IMAGENET1K_V1").to(device)
net = convnext_large(weights="IMAGENET1K_V1").to(device)

############## END OF CODE ##############

<!-- Tip Box -->
<div style="background-color: #E8F8F5; border-left: 5px solid #1ABC9C; padding: 15px; border-radius: 5px; margin: 20px 0;">
    <p style="color: #16A085; margin: 0;"><strong>Tip:</strong> Now you have tried a different size of the ConvNeXt model. You can also follow the same steps above in this notebook to further evaluate your new trained model.</p>
</div>

<div style="background-color: #f2f0fc; padding: 15px; border-radius: 5px; margin: 30px 0;">
    <h3 style="color: #2f1381; border-bottom: 1px solid #2f1381; padding-bottom: 5px;">Conclusion</h3>
    <p style="color: #2f1381;">In this lab, you have:</p>
    <ul>
        <li style="color: #2f1381;">Loaded and transformed the ConvNeXt dataset</li>
        <li style="color: #2f1381;">Fine-tuned the model to fit your data</li>
        <li style="color: #2f1381;">Tested, evaluated, and visualized the results</li>
        <li style="color: #2f1381;">Experimented with different ConvNeXt model sizes</li>
    </ul>
    <h4 style="color: #2f1381; margin-top: 15px;">Additional Resources</h4>
    <ul>
        <li><a href="https://arxiv.org/abs/1412.0623">Material Recognition in the Wild with the Materials in Context Database</a></li>
    </ul>
</div>

<p style="padding: 10px; border: 1px solid black;">
<img src="../common/images/mlu-logo.png" alt="drawing" width="400"/> <br/>

# Thank you!